# ECGR 4105/5105 Midterm Project Template

**Project:** Breast Cancer Classification  
**Group members:** TODO: Name 1, Name 2, Name 3  
**Course:** ECGR 4105/5105 - Introduction to Machine Learning  
**Term:** Summer 2026

This notebook is organized to match the project requirements: load data, preprocess it consistently, train at least five classifiers, report evaluation metrics, compare models, and prepare discussion answers.

## Project Checklist

- [ ] Confirm all group member names.
- [ ] Load the training and validation/test datasets.
- [ ] Apply the same preprocessing steps to training and validation/test data.
- [ ] Train at least five classifiers.
- [ ] Report confusion matrix, accuracy, precision, recall, and F1-score for each classifier.
- [ ] Include a comparison table.
- [ ] Discuss the best-performing classifier using more than one metric.
- [ ] Prepare answers for the discussion questions.

## 1. Setup

Run this cell first. If any package is missing, install it before continuing.

In [ ]:
# Optional: uncomment and run this cell if your Jupyter environment is missing packages.
# %pip install numpy pandas scikit-learn matplotlib

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
plt.style.use("seaborn-v0_8-whitegrid")

## 2. Data Loading

Use the built-in scikit-learn breast cancer dataset unless your instructor provides separate train/test CSV files. If CSV files are provided, set `USE_CSV_DATA = True` and update the paths and target column.

In [ ]:
USE_CSV_DATA = False

TRAIN_CSV_PATH = "data/train.csv"
TEST_CSV_PATH = "data/test.csv"
TARGET_COLUMN = "target"  # common alternatives: "diagnosis", "class", "label"

def normalize_target(series):
    """Map common breast-cancer labels to sklearn's convention: 0=malignant, 1=benign."""
    if pd.api.types.is_numeric_dtype(series):
        return series.astype(int)

    mapping = {
        "m": 0,
        "malignant": 0,
        "0": 0,
        "b": 1,
        "benign": 1,
        "1": 1,
    }
    return series.astype(str).str.strip().str.lower().map(mapping).astype(int)

if USE_CSV_DATA:
    train_df = pd.read_csv(TRAIN_CSV_PATH)
    test_df = pd.read_csv(TEST_CSV_PATH)

    X_train = train_df.drop(columns=[TARGET_COLUMN])
    y_train = normalize_target(train_df[TARGET_COLUMN])
    X_test = test_df.drop(columns=[TARGET_COLUMN])
    y_test = normalize_target(test_df[TARGET_COLUMN])
    feature_names = X_train.columns.tolist()
    target_names = ["malignant", "benign"]
else:
    data = load_breast_cancer(as_frame=True)
    X = data.data
    y = data.target
    feature_names = data.feature_names.tolist()
    target_names = data.target_names.tolist()  # ['malignant', 'benign']

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=y,
    )

print(f"Training shape: {X_train.shape}")
print(f"Validation/test shape: {X_test.shape}")
print("Class labels:", dict(enumerate(target_names)))
pd.Series(y_train).value_counts().sort_index().rename(index=dict(enumerate(target_names)))

## 3. Exploratory Data Check

Use this section to confirm feature types, missing values, class balance, and any issues that affect preprocessing.

In [ ]:
display(X_train.head())

summary = pd.DataFrame({
    "dtype": X_train.dtypes.astype(str),
    "missing_train": X_train.isna().sum(),
    "missing_test": X_test.isna().sum(),
    "mean_train": X_train.mean(numeric_only=True),
    "std_train": X_train.std(numeric_only=True),
})
display(summary)

pd.Series(y_train).value_counts(normalize=True).sort_index().rename(index=dict(enumerate(target_names))).plot(
    kind="bar",
    title="Training Class Distribution",
    ylabel="Proportion",
    rot=0,
)
plt.show()

## 4. Preprocessing Plan

**Document your choices here.**

- Missing values: TODO: describe imputation choice.
- Scaling/standardization: TODO: explain which models need scaling and why.
- Validation policy: the validation/test data must not be used for training or preprocessing fit.

The pipelines below fit imputers/scalers on the training data only, then apply the same learned transformations to the validation/test data.

## 5. Classifier Definitions

This template includes all six classifiers from the assignment. You may report all six, or choose at least five.

In [ ]:
scaled_preprocess = [
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
]

unscaled_preprocess = [
    ("imputer", SimpleImputer(strategy="median")),
]

models = {
    "Logistic Regression": Pipeline(scaled_preprocess + [
        ("model", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))
    ]),
    "Naive Bayes": Pipeline(unscaled_preprocess + [
        ("model", GaussianNB())
    ]),
    "Perceptron": Pipeline(scaled_preprocess + [
        ("model", Perceptron(max_iter=5000, random_state=RANDOM_STATE))
    ]),
    "SVM": Pipeline(scaled_preprocess + [
        ("model", SVC(kernel="rbf", random_state=RANDOM_STATE))
    ]),
    "Decision Tree": Pipeline(unscaled_preprocess + [
        ("model", DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE))
    ]),
    "KNN": Pipeline(scaled_preprocess + [
        ("model", KNeighborsClassifier(n_neighbors=5))
    ]),
}

list(models.keys())

## 6. Train and Evaluate Models

The assignment asks for confusion matrix, accuracy, precision, recall, and F1-score. Because missing malignant cases is especially costly in breast cancer screening, this notebook treats **malignant** as the positive class for precision, recall, and F1.

In [ ]:
POS_LABEL = 0  # sklearn breast cancer labels: 0=malignant, 1=benign
MALIGNANT_F1_SCORER = make_scorer(f1_score, pos_label=POS_LABEL, zero_division=0)

def evaluate_model(name, estimator, X_train, y_train, X_test, y_test):
    estimator.fit(X_train, y_train)
    y_pred = estimator.predict(X_test)

    return {
        "Classifier": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision (malignant)": precision_score(y_test, y_pred, pos_label=POS_LABEL, zero_division=0),
        "Recall (malignant)": recall_score(y_test, y_pred, pos_label=POS_LABEL, zero_division=0),
        "F1-score (malignant)": f1_score(y_test, y_pred, pos_label=POS_LABEL, zero_division=0),
        "Confusion Matrix": confusion_matrix(y_test, y_pred, labels=[0, 1]),
        "Predictions": y_pred,
        "Estimator": estimator,
    }

results = [evaluate_model(name, model, X_train, y_train, X_test, y_test) for name, model in models.items()]

metrics_df = pd.DataFrame(results).drop(columns=["Confusion Matrix", "Predictions", "Estimator"])
metrics_df = metrics_df.sort_values(by=["F1-score (malignant)", "Recall (malignant)", "Accuracy"], ascending=False)
display(metrics_df.style.format({
    "Accuracy": "{:.3f}",
    "Precision (malignant)": "{:.3f}",
    "Recall (malignant)": "{:.3f}",
    "F1-score (malignant)": "{:.3f}",
}))

In [ ]:
for result in results:
    print("=" * 80)
    print(result["Classifier"])
    print(classification_report(y_test, result["Predictions"], target_names=target_names, zero_division=0))

## 7. Confusion Matrices

Use these plots in the notebook and short report. Rows are true labels and columns are predicted labels.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.ravel()

for ax, result in zip(axes, results):
    disp = ConfusionMatrixDisplay(
        confusion_matrix=result["Confusion Matrix"],
        display_labels=target_names,
    )
    disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(result["Classifier"])

for ax in axes[len(results):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

## 8. Scaling Comparison

Use this section to answer how scaling affected Logistic Regression, SVM, KNN, and Decision Tree. Tree splits are generally insensitive to feature scaling, while distance-based and margin/gradient-based methods usually depend on comparable feature scales.

In [ ]:
scaling_models = {
    "Logistic Regression": LogisticRegression(max_iter=5000, random_state=RANDOM_STATE),
    "SVM": SVC(kernel="rbf", random_state=RANDOM_STATE),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE),
}

scaling_rows = []
for name, clf in scaling_models.items():
    for scaled in [False, True]:
        steps = [("imputer", SimpleImputer(strategy="median"))]
        if scaled:
            steps.append(("scaler", StandardScaler()))
        steps.append(("model", clf))
        estimator = Pipeline(steps)
        estimator.fit(X_train, y_train)
        pred = estimator.predict(X_test)
        scaling_rows.append({
            "Classifier": name,
            "Scaled": scaled,
            "Accuracy": accuracy_score(y_test, pred),
            "Recall (malignant)": recall_score(y_test, pred, pos_label=POS_LABEL, zero_division=0),
            "F1-score (malignant)": f1_score(y_test, pred, pos_label=POS_LABEL, zero_division=0),
        })

scaling_df = pd.DataFrame(scaling_rows)
display(scaling_df.style.format({
    "Accuracy": "{:.3f}",
    "Recall (malignant)": "{:.3f}",
    "F1-score (malignant)": "{:.3f}",
}))

## 9. Decision Tree `max_depth` Study

Use cross-validation on the training data to discuss underfitting, overfitting, and validation performance. Do not tune on the final validation/test labels.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

tree_search = GridSearchCV(
    Pipeline(unscaled_preprocess + [("model", DecisionTreeClassifier(random_state=RANDOM_STATE))]),
    param_grid={"model__max_depth": [1, 2, 3, 4, 5, 6, 8, 10, None]},
    scoring=MALIGNANT_F1_SCORER,
    cv=cv,
    return_train_score=True,
)
tree_search.fit(X_train, y_train)

tree_cv = pd.DataFrame(tree_search.cv_results_)[[
    "param_model__max_depth",
    "mean_train_score",
    "mean_test_score",
    "rank_test_score",
]]
tree_cv = tree_cv.rename(columns={
    "param_model__max_depth": "max_depth",
    "mean_train_score": "Mean CV Train F1",
    "mean_test_score": "Mean CV Validation F1",
    "rank_test_score": "Rank",
})
display(tree_cv.sort_values("Rank"))

print("Best max_depth:", tree_search.best_params_["model__max_depth"])
print("Best CV F1:", round(tree_search.best_score_, 3))

## 10. KNN `k` Selection

Use cross-validation on the training data to select `k`. Very small `k` can memorize local noise; very large `k` can smooth away useful class boundaries.

In [ ]:
knn_search = GridSearchCV(
    Pipeline(scaled_preprocess + [("model", KNeighborsClassifier())]),
    param_grid={"model__n_neighbors": [1, 3, 5, 7, 9, 11, 15, 21]},
    scoring=MALIGNANT_F1_SCORER,
    cv=cv,
    return_train_score=True,
)
knn_search.fit(X_train, y_train)

knn_cv = pd.DataFrame(knn_search.cv_results_)[[
    "param_model__n_neighbors",
    "mean_train_score",
    "mean_test_score",
    "rank_test_score",
]]
knn_cv = knn_cv.rename(columns={
    "param_model__n_neighbors": "k",
    "mean_train_score": "Mean CV Train F1",
    "mean_test_score": "Mean CV Validation F1",
    "rank_test_score": "Rank",
})
display(knn_cv.sort_values("Rank"))

print("Best k:", knn_search.best_params_["model__n_neighbors"])
print("Best CV F1:", round(knn_search.best_score_, 3))

## 11. Short Report Draft

Use this section as the report text foundation. Replace TODOs after running the notebook.

### Classifiers Used

TODO: List at least five classifiers used. This template includes Logistic Regression, Naive Bayes, Perceptron, SVM, Decision Tree, and KNN.

### Preprocessing Steps

TODO: Describe data loading, train/validation split or provided train/test files, missing-value handling, and feature scaling.

### Evaluation Results

TODO: Summarize the metrics table and confusion matrices.

### Best-Performing Classifier

TODO: Identify the best model. Define "best" using more than one metric, such as malignant recall, F1-score, and accuracy.

### Notes on Clinical Context

TODO: Explain why recall for malignant cases matters: false negatives may delay diagnosis and treatment.

## 12. Discussion Question Preparation

Use these prompts to prepare for the Monday discussion.

1. **Which model performed best overall?**  
   TODO: Define best and cite multiple metrics.

2. **Which model achieved the highest recall for malignant cases? Why is this important?**  
   TODO: Use the comparison table and explain false negatives.

3. **How did scaling affect Logistic Regression, SVM, KNN, and Decision Tree?**  
   TODO: Refer to the scaling comparison table and algorithm behavior.

4. **For the Decision Tree, how did `max_depth` affect training and validation/cross-validation performance?**  
   TODO: Discuss underfitting, overfitting, pruning, `max_depth`, `min_samples_leaf`, and cross-validation.

5. **How does a Decision Tree select a feature and split point at each node?**  
   TODO: Explain impurity reduction using Gini impurity or entropy.

6. **For KNN, how did you select `k`?**  
   TODO: Explain cross-validation and the bias-variance tradeoff.

7. **For each classifier, explain assumptions, decision boundaries, sensitivities, and failure modes.**  
   TODO: Prepare one concise paragraph per classifier.

## 13. Final Submission Notes

- Restart the kernel and run all cells before submitting.
- Make sure the notebook contains outputs, figures, tables, and concise explanations.
- Confirm the short report includes the comparison table and confusion matrices.
- Do not train on validation/test data.